# Interactive Target4 Landscape

This notebook visualizes the 2D exact cosine target family

$$g(\xi)=c+\sum_{i=1}^2 a_i \cos(2\pi k_i x_i + \phi_i)$$

and the tilted density

$$\nu_\beta(\xi) \propto \exp(\lambda g(\xi))$$

with interactive controls for `lambda`, `c`, `a`, `k`, and `phi`.

The plot is a 2D landscape over $(x_1, x_2) \in [0,1]^2$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import ipywidgets as widgets
from IPython.display import display

from learning.module.gbs.target4_family import Target4HarmonicParams
from learning.module.gbs.target4_notebook_utils import target4_logprob

plt.rcParams['figure.figsize'] = (7, 6)
plt.rcParams['figure.dpi'] = 120

In [2]:
def make_params(c, a1, a2, k1, k2, phi1, phi2):
    total_amp = abs(a1) + abs(a2)
    bound = min(c, 1.0 - c)
    if total_amp > bound:
        scale = bound / max(total_amp, 1e-12)
        a1 = a1 * scale
        a2 = a2 * scale
    return Target4HarmonicParams(
        c=np.float32(c),
        a=np.asarray([a1, a2], dtype=np.float32),
        k=np.asarray([k1, k2], dtype=np.float32),
        phi=np.asarray([phi1, phi2], dtype=np.float32),
    )


def evaluate_landscape(lam, params, n=180):
    x = np.linspace(0.0, 1.0, n)
    y = np.linspace(0.0, 1.0, n)
    X, Y = np.meshgrid(x, y, indexing='xy')
    grid = np.stack([X.reshape(-1), Y.reshape(-1)], axis=-1)
    logp = np.asarray(target4_logprob(grid, lam, target_params=params)).reshape(n, n)
    logp = logp - np.max(logp)
    density = np.exp(logp)
    g = (
        params.c
        + params.a[0] * np.cos(2.0 * np.pi * params.k[0] * X + params.phi[0])
        + params.a[1] * np.cos(2.0 * np.pi * params.k[1] * Y + params.phi[1])
    )
    return X, Y, g, density


def draw_landscape(lam, c, a1, a2, k1, k2, phi1, phi2, mode='density'):
    params = make_params(c, a1, a2, k1, k2, phi1, phi2)
    X, Y, g, density = evaluate_landscape(lam, params)

    fig, ax = plt.subplots(1, 1)
    Z = density if mode == 'density' else g
    title = 'Tilted density' if mode == 'density' else 'Target landscape g(x)'
    contour = ax.contourf(X, Y, Z, levels=24, cmap='viridis')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='5%', pad=0.08)
    fig.colorbar(contour, cax=cax)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(f'{title} | lambda={lam:.2f}, c={float(params.c):.2f}')
    plt.show()

    print('effective amplitudes:', np.asarray(params.a))
    print('k:', np.asarray(params.k))
    print('phi:', np.asarray(params.phi))
    print('max allowed sum |a_i|:', min(c, 1.0 - c))

In [ ]:
lambda_play = widgets.Play(value=0, min=-40, max=40, step=1, interval=120, description='Play')
lambda_slider = widgets.FloatSlider(value=0.0, min=-40.0, max=40.0, step=0.5, description='lambda')
widgets.jslink((lambda_play, 'value'), (lambda_slider, 'value'))

c_slider = widgets.FloatSlider(value=0.75, min=0.05, max=0.95, step=0.01, description='c')
a1_slider = widgets.FloatSlider(value=0.25, min=-0.5, max=0.5, step=0.01, description='a1')
a2_slider = widgets.FloatSlider(value=0.25, min=-0.5, max=0.5, step=0.01, description='a2')
k1_slider = widgets.IntSlider(value=4, min=1, max=8, step=1, description='k1')
k2_slider = widgets.IntSlider(value=6, min=1, max=8, step=1, description='k2')
phi1_slider = widgets.FloatSlider(value=0.0, min=0.0, max=2.0 * np.pi, step=0.05, description='phi1')
phi2_slider = widgets.FloatSlider(value=0.0, min=0.0, max=2.0 * np.pi, step=0.05, description='phi2')
mode_dropdown = widgets.Dropdown(options=['density', 'g'], value='density', description='show')

ui_left = widgets.VBox([lambda_play, lambda_slider, c_slider, mode_dropdown])
ui_right = widgets.VBox([a1_slider, a2_slider, k1_slider, k2_slider, phi1_slider, phi2_slider])
display(widgets.HBox([ui_left, ui_right]))

out = widgets.interactive_output(
    draw_landscape,
    {
        'lam': lambda_slider,
        'c': c_slider,
        'a1': a1_slider,
        'a2': a2_slider,
        'k1': k1_slider,
        'k2': k2_slider,
        'phi1': phi1_slider,
        'phi2': phi2_slider,
        'mode': mode_dropdown,
    },
)
display(out)

Output()

## Notes

- `show='density'` plots the normalized tilted landscape `exp(log p)` up to a max-shift for numerical stability.
- `show='g'` plots the raw target landscape.
- If `|a1| + |a2| > min(c, 1-c)`, the amplitudes are automatically rescaled to keep `g(x)` inside `[0,1]`.